In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report#, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.model_selection import KFold
from sklearn import linear_model

In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv")

In [3]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [5]:
df.shape

(5000, 9)

In [12]:
df.dtypes

lead_source                  object
industry                     object
employment_status            object
location                     object
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [4]:
df.isna().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [6]:
df[df.columns[df.dtypes == 'object']] = df[df.columns[df.dtypes == 'object']].fillna('NA')
numeric_cols = df.select_dtypes(include=['number']).columns
df[numeric_cols] = df[numeric_cols].fillna(0)

In [7]:
# Dataset splitting
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

I have to compute the AUC score for every numerical variable, taking it as the prediction score, and the "converted" variable as the actual value. This tells us how correlated these variables are with the target variable. For variables that don't lie on the [0,1] interval, I linearly transform them by mapping $X_{nor} = (X - X_{min})/(X_{max} - X_{min})$.

In [39]:
for var in df.select_dtypes(include=['number']).columns[:-1]:
    #nor_var = (df_train[var] - df_train[var].min()) / (df_train[var].max() - df_train[var].min())
    #scr_nor = roc_auc_score(df_train.converted, nor_var)
    scr = roc_auc_score(df_train.converted, df_train[var])
    print(f"For {var}: {scr}")#, Normalized: {scr_nor}")

For annual_income: 0.6081977776250831
For number_of_courses_viewed: 0.7229894623989618
For interaction_count: 0.7649203047563227
For lead_score: 0.7882254810906102


<span style="color: orange">Q1. Which numerical variable (among the following 4) has the highest AUC?</span>

<span style="color: lightgreen">Ans.: "lead_score" is the variable with the highest AUC score with the target variable.</span>

Other way 

In [34]:
var = df.select_dtypes(include=['number']).columns[0]
df_train[var].min()

0.0